#  Neo4j test

## 1.test library

In [1]:
import neo4j

print(neo4j.__version__)


6.3.1


## 2. test connection v1

In [2]:
from graph_storage.connection import Neo4jConnection

connection = Neo4jConnection()


try:

    status = connection.verify_connection()


    if status:

        print(
            "Neo4j connection successful ✅"
        )


    else:

        print(
            "Connection failed ❌"
        )


finally:

    connection.close()

Neo4j connection successful ✅


## 3. test conn v3

In [1]:
from graph_storage.connection import Neo4jConnection
connection = Neo4jConnection()


with connection as conn:
    if conn.verify_connection():
        print("Connected to Neo4j successfully!")
    else:
        print("Failed to connect.")

Connected to Neo4j successfully!


## 4. test graph loader v1

In [18]:
from graph_storage.connection import Neo4jConnection
from graph_storage.graph_loader import Neo4jGraphLoader


DOCUMENT_ID = "doc_loader_test_001"


connection = Neo4jConnection(
    "config/neo4j_config.json"
)


loader = Neo4jGraphLoader(
    connection=connection,
    schema_path="schema_registry.json"
)


test_data  = {
    "document_id": "doc_002",

    "entities": [
        {
            "entity_key": "entity_1",
            "name": "مهندس رضایی",
            "type": "Person",
            "properties": {}
        },
        {
            "entity_key": "entity_2",
            "name": "شرکت الکتروپارس",
            "type": "Organization",
            "properties": {}
        }
    ],

    "relationships": [
        {
            "source": "entity_1",
            "target": "entity_2",
            "type": "WORKS_AT",
            "properties": {
                "":
                    "مهندس رضایی در شرکت الکتروپارس فعالیت می‌کند."
            }
        }
    ]
}


try:

    # -----------------------------------
    # فقط داده همین تست قبلی را پاک می‌کند
    # -----------------------------------

    with connection.driver.session(
        database=connection.config.get(
            "database",
            "neo4j"
        )
    ) as session:

        session.run(
            """
            MATCH (n {document_id: $document_id})
            DETACH DELETE n
            """,
            document_id=DOCUMENT_ID
        ).consume()


    # -----------------------------------
    # اولین Load
    # -----------------------------------

    result_1 = loader.load_document(
        test_data
    )

    print(
        "First load:",
        result_1
    )


    # -----------------------------------
    # دوباره همان سند را Load می‌کنیم
    # تا MERGE را تست کنیم
    # -----------------------------------

    result_2 = loader.load_document(
        test_data
    )

    print(
        "Second load:",
        result_2
    )


    # -----------------------------------
    # بررسی نتیجه داخل Neo4j
    # -----------------------------------

    with connection.driver.session(
        database=connection.config.get(
            "database",
            "neo4j"
        )
    ) as session:

        node_record = session.run(
            """
            MATCH (n {document_id: $document_id})

            RETURN count(n) AS node_count
            """,
            document_id=DOCUMENT_ID
        ).single()


        relationship_record = session.run(
            """
            MATCH
                (a {document_id: $document_id})
                -[r]->
                (b {document_id: $document_id})

            RETURN count(r) AS relationship_count
            """,
            document_id=DOCUMENT_ID
        ).single()


        print(
            "Node count:",
            node_record["node_count"]
        )

        print(
            "Relationship count:",
            relationship_record[
                "relationship_count"
            ]
        )


        records = session.run(
            """
            MATCH
                (a {document_id: $document_id})
                -[r]->
                (b {document_id: $document_id})

            RETURN
                labels(a) AS source_labels,
                a.name AS source,
                type(r) AS relationship,
                r.evidence AS evidence,
                labels(b) AS target_labels,
                b.name AS target
            """,
            document_id=DOCUMENT_ID
        )

        print("\nGraph:")

        for record in records:

            print(
                record["source_labels"],
                record["source"],
                "--",
                record["relationship"],
                "-->",
                record["target_labels"],
                record["target"],
            )

            print(
                "Evidence:",
                record["evidence"]
            )


finally:

    connection.close()

First load: {'document_id': 'doc_002', 'entities_processed': 2, 'relationships_processed': 1}
Second load: {'document_id': 'doc_002', 'entities_processed': 2, 'relationships_processed': 1}
Node count: 0
Relationship count: 0

Graph:


In [1]:
from graph_storage.connection import Neo4jConnection
from graph_storage.graph_loader import Neo4jGraphLoader


# ============================================================
# Test Document 1
# ============================================================

document_1 = {
    "document_id": "test_doc_001",

    "entities": [
        {
            "entity_key": "entity_1",
            "name": "دکتر سهرابی",
            "type": "Person",
            "properties": {
                "role": "مدیرعامل",
                "aliases": [
                    "دکتر سهرابی",
                    "خود او"
                ]
            }
        },

        {
            "entity_key": "entity_2",
            "name": "شرکت خورشید تابان",
            "type": "Organization",
            "properties": {
                "aliases": [
                    "خورشید تابان"
                ]
            }
        }
    ],

    "relationships": [
        {
            "source": "entity_1",
            "target": "entity_2",
            "type": "HEAD_OF",
            "properties": {
                "role": "مدیرعامل",
                "since": "۱۳۹۹",
                "until": "۱۴۰۲",
                "evidence": (
                    "دکتر سهرابی از سال ۱۳۹۹ تا ۱۴۰۲ "
                    "مدیرعامل شرکت خورشید تابان بود."
                )
            }
        }
    ]
}


# ============================================================
# Test Document 2
# ============================================================

document_2 = {
    "document_id": "test_doc_002",

    "entities": [
        {
            "entity_key": "entity_1",
            "name": "مهندس نوروزی",
            "type": "Person",
            "properties": {
                "role": "معاون فنی"
            }
        },

        {
            "entity_key": "entity_2",
            "name": "شرکت آریا انرژی",
            "type": "Organization",
            "properties": {}
        }
    ],

    "relationships": [
        {
            "source": "entity_1",
            "target": "entity_2",
            "type": "WORKS_AT",
            "properties": {
                "role": "معاون فنی",
                "evidence": (
                    "مهندس نوروزی به عنوان معاون فنی "
                    "در شرکت آریا انرژی فعالیت می‌کند."
                )
            }
        }
    ]
}


# ============================================================
# Helper: inspect current graph
# ============================================================

def inspect_graph(connection):
    """
    Print all currently stored nodes and relationships.
    """

    database = connection.config.get(
        "database",
        "neo4j"
    )

    with connection.driver.session(
        database=database
    ) as session:

        # ----------------------------------------------------
        # Nodes
        # ----------------------------------------------------

        node_records = session.run(
            """
            MATCH (n)

            RETURN
                labels(n) AS labels,
                n.document_id AS document_id,
                n.entity_key AS entity_key,
                n.name AS name,
                properties(n) AS properties

            ORDER BY n.entity_key
            """
        ).data()

        # ----------------------------------------------------
        # Relationships
        # ----------------------------------------------------

        relationship_records = session.run(
            """
            MATCH (source)-[r]->(target)

            RETURN
                source.name AS source,
                type(r) AS relationship,
                target.name AS target,
                properties(r) AS properties

            ORDER BY source.name
            """
        ).data()


    print("\n------------------------------")
    print("NODES")
    print("------------------------------")

    if not node_records:
        print("No nodes found.")

    for node in node_records:

        print(
            f"Labels: {node['labels']}\n"
            f"Document: {node['document_id']}\n"
            f"Entity key: {node['entity_key']}\n"
            f"Name: {node['name']}\n"
            f"Properties: {node['properties']}\n"
        )


    print("------------------------------")
    print("RELATIONSHIPS")
    print("------------------------------")

    if not relationship_records:
        print("No relationships found.")

    for relationship in relationship_records:

        print(
            f"{relationship['source']} "
            f"-[{relationship['relationship']}]-> "
            f"{relationship['target']}"
        )

        print(
            "Properties:",
            relationship["properties"]
        )

        print()


    return {
        "nodes": node_records,
        "relationships": relationship_records,
    }


# ============================================================
# Test
# ============================================================

connection = Neo4jConnection()

loader = Neo4jGraphLoader(
    connection=connection,
    schema_path="schema_registry.json",
)


try:

    # --------------------------------------------------------
    # Verify connection
    # --------------------------------------------------------

    if not connection.verify_connection():
        raise RuntimeError(
            "Could not connect to Neo4j."
        )

    print("Neo4j connection successful ✅")


    # ========================================================
    # Load Document 1
    # ========================================================

    print("\n======================================")
    print("LOADING DOCUMENT 1")
    print("======================================")

    result_1 = loader.replace_document(
        document_1
    )

    print(
        "Loader result:",
        result_1
    )

    graph_1 = inspect_graph(
        connection
    )

    assert len(
        graph_1["nodes"]
    ) == 2

    assert len(
        graph_1["relationships"]
    ) == 1

    assert all(
        node["document_id"] == "test_doc_001"
        for node in graph_1["nodes"]
    )

    print(
        "Document 1 test passed ✅"
    )


    # ========================================================
    # Load Document 2
    # ========================================================

    print("\n======================================")
    print("LOADING DOCUMENT 2")
    print("======================================")

    result_2 = loader.replace_document(
        document_2
    )

    print(
        "Loader result:",
        result_2
    )

    graph_2 = inspect_graph(
        connection
    )

    # --------------------------------------------------------
    # Important:
    # document 1 must be completely gone.
    # --------------------------------------------------------

    assert len(
        graph_2["nodes"]
    ) == 2

    assert len(
        graph_2["relationships"]
    ) == 1

    assert all(
        node["document_id"] == "test_doc_002"
        for node in graph_2["nodes"]
    )

    assert not any(
        node["document_id"] == "test_doc_001"
        for node in graph_2["nodes"]
    )

    print(
        "Document replacement test passed ✅"
    )


    print("\n======================================")
    print("ALL GRAPH LOADER TESTS PASSED ✅")
    print("======================================")


finally:

    connection.close()

Neo4j connection successful ✅

LOADING DOCUMENT 1
Loader result: {'document_id': 'test_doc_001', 'deleted_nodes': 2, 'deleted_relationships': 1, 'entities_loaded': 2, 'relationships_loaded': 1}

------------------------------
NODES
------------------------------
Labels: ['Person']
Document: test_doc_001
Entity key: entity_1
Name: دکتر سهرابی
Properties: {'entity_key': 'entity_1', 'role': 'مدیرعامل', 'aliases': ['دکتر سهرابی', 'خود او'], 'name': 'دکتر سهرابی', 'document_id': 'test_doc_001'}

Labels: ['Organization']
Document: test_doc_001
Entity key: entity_2
Name: شرکت خورشید تابان
Properties: {'entity_key': 'entity_2', 'aliases': ['خورشید تابان'], 'name': 'شرکت خورشید تابان', 'document_id': 'test_doc_001'}

------------------------------
RELATIONSHIPS
------------------------------
دکتر سهرابی -[HEAD_OF]-> شرکت خورشید تابان
Properties: {'role': 'مدیرعامل', 'evidence': 'دکتر سهرابی از سال ۱۳۹۹ تا ۱۴۰۲ مدیرعامل شرکت خورشید تابان بود.', 'until': '۱۴۰۲', 'since': '۱۳۹۹'}

Document 1 test 